# Road Following Live (Autonomous Simulator Mode)

This notebook runs the trained ResNet-18 Road Following model autonomously inside the **DonkeyCar Simulator (`CarSimulator`)**.

### 1. Setup Environment & Load Trained Model

In [ ]:
import os
import sys
from pathlib import Path
import torch
import torchvision

# Add simulation root to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from utils import preprocess
from xy_dataset import XYDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model architecture (ResNet-18)
model = torchvision.models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 2)  # x, y coordinates

model_path = 'road_following_model.pth'
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Successfully loaded model weights from '{model_path}'")
else:
    print(f"Warning: '{model_path}' not found! Please train and save model first in interactive_regression.ipynb")

model = model.to(device).eval()


### 2. Initialize Simulator (`CarSimulator`)

In [ ]:
from simulation import CarSimulator, bgr8_to_jpeg

# Reuse or close previous CarSimulator instance
if 'Car' in globals() and Car is not None:
    try:
        Car.close()
    except Exception:
        pass

Car = CarSimulator()
print("Simulator initialized successfully!")


### 3. Basic Controller (P-Gain + Bias)

In [ ]:
import cv2
import ipywidgets
import traitlets
import threading
import time
from IPython.display import display
from ipywidgets import Layout

slider_style = {'description_width': '140px'}

# Control Sliders with numeric readout display enabled
network_output_slider = ipywidgets.FloatSlider(description='Network Output', min=-1.0, max=1.0, value=0.0, step=0.01, readout=True, readout_format='.2f', disabled=True, layout=Layout(width='400px'), style=slider_style)
steering_gain_slider  = ipywidgets.FloatSlider(description='Steering Gain', min=-2.0, max=2.0, value=1.0, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)
steering_bias_slider  = ipywidgets.FloatSlider(description='Steering Bias', min=-0.5, max=0.5, value=0.0, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)
steering_value_slider = ipywidgets.FloatSlider(description='Final Steering', min=-1.0, max=1.0, value=0.0, step=0.01, readout=True, readout_format='.2f', disabled=True, layout=Layout(width='400px'), style=slider_style)
throttle_slider       = ipywidgets.FloatSlider(description='Throttle', min=-1.0, max=1.0, value=0.15, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)

# Live Stream & Prediction Preview Widget
state_widget = ipywidgets.ToggleButtons(options=['Off', 'Autonomous Live'], description='Mode', value='Off')
prediction_widget = ipywidgets.Image(value=Car.value, format='jpeg', width=Car.obs.shape[1], height=Car.obs.shape[0])
reset_button = ipywidgets.Button(description='Reset Track', button_style='warning', icon='refresh')

live_active = False

def live_drive_loop():
    global live_active
    while live_active:
        try:
            image = Car.obs
            preprocessed = preprocess(image)
            
            with torch.no_grad():
                output = model(preprocessed).detach().cpu().numpy().flatten()
            
            x = float(output[0])
            y = float(output[1]) if len(output) > 1 else 0.0
            
            network_output_slider.value = x
            
            # Calculate final steering value with Gain & Bias
            steering = x * steering_gain_slider.value + steering_bias_slider.value
            steering = max(-1.0, min(1.0, steering))
            steering_value_slider.value = steering
            
            # Step the car in DonkeyCar simulator
            Car.run(steering, throttle_slider.value)
            
            # Draw predicted target dot (blue circle) on video preview
            px = int(Car.obs.shape[1] * (x / 2.0 + 0.5))
            py = int(Car.obs.shape[0] * (y / 2.0 + 0.5))
            
            prediction = Car.obs.copy()
            prediction = cv2.circle(prediction, (px, py), 8, (255, 0, 0), 3)
            prediction_widget.value = bgr8_to_jpeg(prediction)
            
        except Exception as e:
            print(f"Error in live drive loop: {e}")
            break
            
        time.sleep(0.05)  # 20 FPS loop

def on_state_change(change):
    global live_active
    if change['new'] == 'Autonomous Live':
        if not live_active:
            live_active = True
            t = threading.Thread(target=live_drive_loop, daemon=True)
            t.start()
    else:
        live_active = False

state_widget.observe(on_state_change, names='value')

def on_reset_clicked(b):
    global live_active
    state_widget.value = 'Off'
    live_active = False
    time.sleep(0.1)
    Car.reset()
    prediction_widget.value = Car.value

reset_button.on_click(on_reset_clicked)

# Clean Non-Overlapping Layout
center_box = ipywidgets.VBox([
    prediction_widget,
    ipywidgets.HBox([state_widget, reset_button])
], layout=Layout(align_items='center', margin='0px 0px 15px 0px'))

sliders_box = ipywidgets.VBox([
    network_output_slider,
    steering_gain_slider,
    steering_bias_slider,
    steering_value_slider,
    throttle_slider
], layout=Layout(align_items='center'))

display(ipywidgets.VBox([center_box, sliders_box]))


### 4. Full PID Controller with Low-Pass Filtering & Adaptive Cornering Throttle

This advanced controller includes:
* **Proportional ($K_p$):** Direct steering response to predicted target $x$.
* **Integral ($K_i$):** Eliminates steady-state drift on long curves.
* **Derivative ($K_d$):** Dampens wobbles/oscillations ($\frac{de}{dt}$) on straight paths.
* **Low-Pass Filter ($\alpha$):** Smooths out frame-to-frame neural network noise.
* **Adaptive Throttle:** Automatically reduces throttle when making sharp turns ($|\text{steering}|$ is high) to prevent skidding out of track.

In [ ]:
from simulate.PID import PIDController
import cv2
import ipywidgets
import traitlets
import threading
import time
import json
from IPython.display import display
from ipywidgets import Layout


# Interactive PID Sliders with readout=True
kp_slider = ipywidgets.FloatSlider(description='Kp (Gain)', min=0.0, max=3.0, value=1.0, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
ki_slider = ipywidgets.FloatSlider(description='Ki (Integral)', min=0.0, max=0.5, value=0.0, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
kd_slider = ipywidgets.FloatSlider(description='Kd (Damping)', min=0.0, max=1.0, value=0.15, step=0.02, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
alpha_slider = ipywidgets.FloatSlider(description='Alpha (Filter)', min=0.1, max=1.0, value=0.7, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
pid_bias_slider = ipywidgets.FloatSlider(description='Steering Bias', min=-0.5, max=0.5, value=0.0, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)

base_throttle_slider = ipywidgets.FloatSlider(description='Base Throttle', min=0.05, max=0.5, value=0.20, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
brake_gain_slider = ipywidgets.FloatSlider(description='Brake Gain', min=0.0, max=0.4, value=0.10, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)

pid_steering_disp = ipywidgets.FloatSlider(description='Live Steering', min=-1.0, max=1.0, value=0.0, step=0.01, readout=True, readout_format='.2f', disabled=True, layout=Layout(width='400px'), style=pid_style)
pid_throttle_disp = ipywidgets.FloatSlider(description='Live Throttle', min=0.0, max=0.5, value=0.15, step=0.01, readout=True, readout_format='.2f', disabled=True, layout=Layout(width='400px'), style=pid_style)

pid_state_widget = ipywidgets.ToggleButtons(options=['Off', 'PID Live Drive'], description='PID Mode', value='Off')
pid_prediction_widget = ipywidgets.Image(value=Car.value, format='jpeg', width=Car.obs.shape[1], height=Car.obs.shape[0])
pid_reset_button = ipywidgets.Button(description='Reset Track & PID', button_style='warning', icon='refresh')
load_best_btn = ipywidgets.Button(description='Load Best Config', button_style='info', icon='download')

pid_live_active = False

pid = PIDController()

pid_style = {'description_width': '140px'}

def load_best_config(b=None):
    cfg_path = 'best_pid_config.json'
    if os.path.exists(cfg_path):
        with open(cfg_path, 'r', encoding='utf-8') as f:
            cfg = json.load(f)
        kp_slider.value = cfg.get('kp', kp_slider.value)
        ki_slider.value = cfg.get('ki', ki_slider.value)
        kd_slider.value = cfg.get('kd', kd_slider.value)
        alpha_slider.value = cfg.get('alpha', alpha_slider.value)
        pid_bias_slider.value = cfg.get('bias', pid_bias_slider.value)
        base_throttle_slider.value = cfg.get('base_throttle', base_throttle_slider.value)
        brake_gain_slider.value = cfg.get('brake_gain', brake_gain_slider.value)
        print(f"Loaded best config from '{cfg_path}'! Score: {cfg.get('score', 0):.2f}")
    else:
        print(f"Config file '{cfg_path}' not found!")

load_best_btn.on_click(load_best_config)

def pid_drive_loop():
    global pid_live_active
    pid.reset()
    while pid_live_active:
        try:
            image = Car.obs
            preprocessed = preprocess(image)
            
            with torch.no_grad():
                output = model(preprocessed).detach().cpu().numpy().flatten()
            
            raw_x = float(output[0])
            raw_y = float(output[1]) if len(output) > 1 else 0.0
            
            # Compute PID Steering
            steering = pid.update(
                raw_x=raw_x,
                kp=kp_slider.value,
                ki=ki_slider.value,
                kd=kd_slider.value,
                alpha=alpha_slider.value,
                bias=pid_bias_slider.value
            )
            
            # Compute Adaptive Throttle (slow down in sharp turns)
            dyn_throttle = base_throttle_slider.value - brake_gain_slider.value * abs(steering)
            dyn_throttle = max(0.05, min(0.5, dyn_throttle))
            
            pid_steering_disp.value = steering
            pid_throttle_disp.value = dyn_throttle
            
            # Step DonkeyCar simulator
            Car.run(steering, dyn_throttle)
            
            # Draw prediction dot (green circle for PID target)
            px = int(Car.obs.shape[1] * (pid.smoothed_x / 2.0 + 0.5))
            py = int(Car.obs.shape[0] * (raw_y / 2.0 + 0.5))
            
            prediction = Car.obs.copy()
            prediction = cv2.circle(prediction, (px, py), 8, (0, 255, 0), 3)
            pid_prediction_widget.value = bgr8_to_jpeg(prediction)
            
        except Exception as e:
            print(f"Error in PID drive loop: {e}")
            break
            
        time.sleep(0.05)  # 20 FPS loop

def on_pid_state_change(change):
    global pid_live_active
    if change['new'] == 'PID Live Drive':
        if not pid_live_active:
            pid_live_active = True
            t = threading.Thread(target=pid_drive_loop, daemon=True)
            t.start()
    else:
        pid_live_active = False

pid_state_widget.observe(on_pid_state_change, names='value')

def on_pid_reset_clicked(b):
    global pid_live_active
    pid_state_widget.value = 'Off'
    pid_live_active = False
    time.sleep(0.1)
    pid.reset()
    Car.reset()
    pid_prediction_widget.value = Car.value

pid_reset_button.on_click(on_pid_reset_clicked)

# Clean Non-Overlapping Layout
pid_center_box = ipywidgets.VBox([
    pid_prediction_widget,
    ipywidgets.HBox([pid_state_widget, pid_reset_button, load_best_btn])
], layout=Layout(align_items='center', margin='0px 0px 15px 0px'))

pid_tuning_box = ipywidgets.VBox([
    ipywidgets.HTML(value="<h4>PID Steering Tuning</h4>"),
    kp_slider,
    ki_slider,
    kd_slider,
    alpha_slider,
    pid_bias_slider
], layout=Layout(margin='0px 20px 0px 0px'))

throttle_tuning_box = ipywidgets.VBox([
    ipywidgets.HTML(value="<h4>Dynamic Throttle & Outputs</h4>"),
    base_throttle_slider,
    brake_gain_slider,
    ipywidgets.HTML(value="<b>Live Outputs:</b>"),
    pid_steering_disp,
    pid_throttle_disp
])

controls_grid = ipywidgets.HBox([pid_tuning_box, throttle_tuning_box], layout=Layout(justify_content='space-around'))

display(ipywidgets.VBox([pid_center_box, controls_grid]))


### 5. Bayesian Hyperparameter Optimizer (Optuna TPE)

Grid Search with **Optuna Tree-structured Parzen Estimator (TPE)** Bayesian optimization.



In [ ]:
import json
import time
import os
import torch
import optuna
import numpy as np
import ipywidgets
from IPython.display import display, clear_output

# Tuning settings
N_TRIALS          = 60    # Total Optuna trials to run
EPISODES_PER_TRIAL = 3   # Runs per config (averaged to reduce noise)
MAX_STEPS         = 200  # Max simulator steps per episode
BASE_THROTTLE     = 0.20
CONFIG_SAVE_PATH  = 'best_pid_config.json'

optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress verbose logs

# Progress widgets
trial_progress = ipywidgets.IntProgress(
    min=0, max=N_TRIALS, description='Trials:',
    bar_style='info', layout=ipywidgets.Layout(width='450px'),
    style={'description_width': '60px'})

best_score_label = ipywidgets.Label(value='Best score: --')
best_params_out  = ipywidgets.Output()
trial_log_out    = ipywidgets.Output(layout=ipywidgets.Layout(height='200px', overflow_y='auto', border='1px solid #ddd'))

display(ipywidgets.VBox([
    ipywidgets.HBox([trial_progress, best_score_label]),
    ipywidgets.HTML('<b>Best config so far:</b>'),
    best_params_out,
    ipywidgets.HTML('<b>Trial log:</b>'),
    trial_log_out
]))


# Evaluation function 
def evaluate_pid(kp, ki, kd, alpha, bias, brake_gain, trial=None):
    """
    Runs EPISODES_PER_TRIAL independent episodes and returns the mean score.
    Score = -mean_abs_x_deviation * 10 + survival_bonus
    (Lower deviation = better; longer survival = bonus)
    """
    episode_scores = []

    for ep in range(EPISODES_PER_TRIAL):
        Car.reset()
        pid.reset()

        x_deviations = []
        steps = 0

        for step in range(MAX_STEPS):
            image = Car.obs
            preprocessed = preprocess(image)

            with torch.no_grad():
                output = model(preprocessed).detach().cpu().numpy().flatten()

            raw_x = float(output[0])
            x_deviations.append(abs(raw_x))

            steering   = pid.update(raw_x=raw_x, kp=kp, ki=ki, kd=kd, alpha=alpha, bias=bias)
            dyn_throttle = max(0.05, min(0.5, BASE_THROTTLE - brake_gain * abs(steering)))

            result     = Car.run(steering, dyn_throttle)
            terminated = result.get('terminated', False)
            truncated  = result.get('truncated', False)
            steps += 1

            # Optuna intermediate pruning: report after 1st episode mid-point
            if trial is not None and ep == 0 and step == MAX_STEPS // 2:
                intermediate = -float(np.mean(x_deviations)) * 10 + steps * 0.3
                trial.report(intermediate, step)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()

            if terminated or truncated:
                break

        mean_dev = float(np.mean(x_deviations)) if x_deviations else 1.0
        # Score: penalize deviation, reward long survival
        # Negate because Optuna minimizes by default (we maximize score)
        ep_score = -mean_dev * 10 + steps * 0.3
        episode_scores.append(ep_score)

    return float(np.mean(episode_scores))


# Optuna objective
best_so_far = {'score': -float('inf'), 'config': None}

def objective(trial):
    kp         = trial.suggest_float('kp',         0.3,  2.5)
    ki         = trial.suggest_float('ki',         0.0,  0.3)
    kd         = trial.suggest_float('kd',         0.0,  0.5)
    alpha      = trial.suggest_float('alpha',      0.3,  1.0)
    bias       = trial.suggest_float('bias',      -0.2,  0.2)
    brake_gain = trial.suggest_float('brake_gain', 0.0,  0.35)

    score = evaluate_pid(kp, ki, kd, alpha, bias, brake_gain, trial=trial)

    trial_progress.value = trial.number + 1

    with trial_log_out:
        print(f"Trial {trial.number+1:03d}/{N_TRIALS} | "
              f"Kp={kp:.2f} Ki={ki:.2f} Kd={kd:.2f} "
              f"α={alpha:.2f} bias={bias:.2f} brake={brake_gain:.2f} "
              f"→ score={score:.3f}")

    # Track best
    if score > best_so_far['score']:
        best_so_far['score'] = score
        best_so_far['config'] = dict(
            kp=kp, ki=ki, kd=kd, alpha=alpha,
            bias=bias, base_throttle=BASE_THROTTLE,
            brake_gain=brake_gain, score=score
        )
        best_score_label.value = f'Best score: {score:.3f}'
        with best_params_out:
            clear_output(wait=True)
            print(json.dumps(best_so_far['config'], indent=2))

        # Save immediately whenever we find a better config
        with open(CONFIG_SAVE_PATH, 'w', encoding='utf-8') as f:
            json.dump(best_so_far['config'], f, indent=2)

    # Optuna minimizes → negate
    return -score


# Run study
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=50)
)

print(f"Starting Optuna Bayesian Optimization: {N_TRIALS} trials × {EPISODES_PER_TRIAL} episodes each")
print(f"Search space: Kp∈[0.3,2.5], Ki∈[0,0.3], Kd∈[0,0.5], α∈[0.3,1.0], bias∈[-0.2,0.2], brake∈[0,0.35]\n")

study.optimize(objective, n_trials=N_TRIALS, catch=(Exception,))

# Print final summary
print("\n" + "="*60)
print(f" Optimization complete! Best trial: #{study.best_trial.number + 1}")
print(f"   Score (negated Optuna value): {-study.best_value:.4f}")
print(json.dumps(best_so_far['config'], indent=2))
print(f"\nBest config saved to '{CONFIG_SAVE_PATH}'")
print("Click 'Load Best Config' in Section 4 to apply it to the PID sliders.")


Starting Optuna Bayesian Optimization: 60 trials × 3 episodes each
Search space: Kp∈[0.3,2.5], Ki∈[0,0.3], Kd∈[0,0.5], α∈[0.3,1.0], bias∈[-0.2,0.2], brake∈[0,0.35]



INFO:simulation:steering=-0.33 | throttle=0.18 | reward=0.000
INFO:simulation:steering=-0.76 | throttle=0.16 | reward=0.000
INFO:simulation:steering=-0.29 | throttle=0.18 | reward=0.006
INFO:simulation:steering=-0.25 | throttle=0.19 | reward=0.032
INFO:simulation:steering=-0.31 | throttle=0.18 | reward=0.048
INFO:simulation:steering=-0.36 | throttle=0.18 | reward=0.074
INFO:simulation:steering=-0.32 | throttle=0.18 | reward=0.099
INFO:simulation:steering=-0.32 | throttle=0.18 | reward=0.121
INFO:simulation:steering=-0.20 | throttle=0.19 | reward=0.143
INFO:simulation:steering=-0.21 | throttle=0.19 | reward=0.165
INFO:simulation:steering=-0.20 | throttle=0.19 | reward=0.187
INFO:simulation:steering=-0.28 | throttle=0.18 | reward=0.210
INFO:simulation:steering=-0.29 | throttle=0.18 | reward=0.231
INFO:simulation:steering=-0.26 | throttle=0.19 | reward=0.253
INFO:simulation:steering=-0.28 | throttle=0.18 | reward=0.274
INFO:simulation:steering=-0.20 | throttle=0.19 | reward=0.295
INFO:sim


✅ Optimization complete! Best trial: #3
   Score (negated Optuna value): 58.0602
{
  "kp": 2.131373809760928,
  "ki": 0.06370173320348284,
  "kd": 0.09091248360355031,
  "alpha": 0.4283831568974037,
  "bias": -0.07830310281618491,
  "base_throttle": 0.2,
  "brake_gain": 0.18366475107128324,
  "score": 58.060159469928486
}

Best config saved to 'best_pid_config.json'
Click 'Load Best Config' in Section 4 to apply it to the PID sliders.
